# Avaliacao de Rotas Medicas

Este notebook apresenta a base de avaliacao do problema de otimizacao de rotas medicas para distribuicao de insumos hospitalares.

A implementacao demonstrada aqui cobre:

- carregamento do catalogo de pontos;
- calculo da matriz de distancias por Haversine;
- decodificacao de cromossomos em rotas reais;
- insercao automatica de abastecimento;
- calculo da fitness da rota.

Configuração inicial para carregar os módulos do projeto.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_CONFIG
from src.data_loader import get_hospitals, get_origin, get_supply_stations, load_points
from src.distance import build_distance_matrix
from src.fitness import evaluate
from src.models import Config

## 1. Catalogo de pontos

A massa de dados possui uma origem, hospitais obrigatorios e unidades de abastecimento. O cromossomo deve conter apenas indices de hospitais.

In [ ]:
points = load_points(PROJECT_ROOT / "data" / "pontos_entrega.csv")
origin = get_origin(points)
hospitals = get_hospitals(points)
supply_stations = get_supply_stations(points)

print(f"Origem: {origin.idx} - {origin.name}")
print(f"Hospitais obrigatorios: {len(hospitals)}")
print(f"Unidades de abastecimento: {len(supply_stations)}")
print(f"Total de pontos: {len(points)}")

In [ ]:
def show_points_table(points):
    header = f"{'idx':>3} | {'tipo':<8} | {'nome':<24} | {'prioridade':<10} | {'demanda':>7}"
    print(header)
    print('-' * len(header))
    for point in points:
        priority = point.priority or '-'
        print(f"{point.idx:>3} | {point.type:<8} | {point.name:<24} | {priority:<10} | {point.demand:>7}")

show_points_table(points)

## 2. Matriz de distancias

A matriz de distancias e calculada uma vez e depois reutilizada pelas avaliacoes. Cada distancia fica indexada pelo par `(origem, destino)`.

In [ ]:
distance_matrix = build_distance_matrix(points)

print(f"Pares calculados: {len(distance_matrix)}")
print(f"Distancia 0 -> 1: {distance_matrix[(0, 1)]:.2f} km")
print(f"Distancia 1 -> 0: {distance_matrix[(1, 0)]:.2f} km")
print(f"Distancia 0 -> 0: {distance_matrix[(0, 0)]:.2f} km")

## 3. Funcao auxiliar para exibir resultados

A funcao abaixo organiza os principais campos retornados por `evaluate()`.

In [ ]:
def describe_route(route, points):
    names_by_idx = {point.idx: point.name for point in points}
    return " -> ".join(f"{idx} ({names_by_idx[idx]})" for idx in route)


def print_result(title, result):
    print(title)
    print('-' * len(title))
    print(f"Cromossomo: {result.chromosome}")
    print(f"Rota decodificada: {result.decoded_route}")
    if result.decoded_route:
        print(f"Rota com nomes: {describe_route(result.decoded_route, points)}")
    print(f"Distancia total: {result.total_distance:.2f} km")
    print(f"Penalidade de prioridade: {result.priority_penalty:.2f}")
    print(f"Penalidade de abastecimento: {result.supply_penalty:.2f}")
    print(f"Reabastecimentos: {result.resupply_count}")
    print(f"Fitness: {result.fitness:.2f}")
    print(f"Valido: {result.is_valid}")
    if result.errors:
        print(f"Erros: {result.errors}")

## 4. Avaliacao de uma rota candidata

O exemplo abaixo mostra um cromossomo simples. A rota real inclui a origem e pode incluir abastecimento automaticamente.

In [ ]:
chromosome = [3, 1, 5, 2, 4]
result = evaluate(chromosome, points, distance_matrix, DEFAULT_CONFIG)

print_result("Avaliacao principal", result)

## 5. Comparacao de prioridade

A penalidade de prioridade usa pesos `ALTA=3`, `MEDIA=2` e `BAIXA=1`. Como a penalidade multiplica peso por posicao de visita, hospitais de prioridade alta tendem a produzir menor penalidade quando aparecem mais cedo.

In [ ]:
route_high_first = evaluate([1, 3, 2], points, distance_matrix, DEFAULT_CONFIG)
route_high_later = evaluate([3, 2, 1], points, distance_matrix, DEFAULT_CONFIG)

print_result("Prioridade ALTA mais cedo", route_high_first)
print()
print_result("Prioridade ALTA mais tarde", route_high_later)
print()
print(f"Diferenca de penalidade: {route_high_later.priority_penalty - route_high_first.priority_penalty:.2f}")

## 6. Comparacao de abastecimento

Com capacidade menor, a rota precisa inserir abastecimentos. Isso afeta a distancia percorrida e a penalidade operacional de abastecimento.

In [ ]:
low_capacity_config = Config(vehicle_capacity=50, lambda_priority=5.0, lambda_supply=10.0)

without_extra_supply = evaluate([7], points, distance_matrix, low_capacity_config)
with_supply = evaluate([7, 10], points, distance_matrix, low_capacity_config)

print_result("Uma entrega com capacidade reduzida", without_extra_supply)
print()
print_result("Duas entregas com capacidade reduzida", with_supply)

## 7. Validacao de cromossomo

A funcao `evaluate()` retorna `EvaluationResult` tambem para entradas invalidas. Nesse caso, `is_valid=False` e a lista `errors` descreve os problemas encontrados.

In [ ]:
invalid_chromosome = [0, 1, 1, 100]
invalid_result = evaluate(invalid_chromosome, points, distance_matrix, DEFAULT_CONFIG)

print_result("Cromossomo invalido", invalid_result)

## 8. Uso direto da funcao `evaluate()`

A chamada principal do projeto e:

```python
result = evaluate(chromosome, points, distance_matrix, config)
fitness = result.fitness
```

O resultado tambem traz a rota decodificada, distancia total, penalidade de prioridade, penalidade de abastecimento, quantidade de reabastecimentos e erros de validacao.